# Pseudotime calculation: Inhibitory neurons lineage (Astrocytes excluded)

Computes Diffusion Pseudo-Time (DPT) on the gene-expression UMI matrix, restricted to the inhibitory lineage minus Astrocytes, of the Knoblich organoid dataset. Output: a `dpt_pseudotime` column attached to a lineage-subset checkpoint AnnData saved to `cellranger/aggregated_annotated_fix_filtered_normalized_inhibitory_dpt.h5ad`.

Astrocytes are excluded from the lineage subset to mirror the published Knoblich CHOOSE paper (Li et al. 2023, Nature). Verification: their reproducibility scripts at github.com/quadbio/ASD_CHOOSE/blob/main/scripts/fig2.R explicitly assign Astrocytes to `state == 'other'` and `lineage == 'other'` (`crop$celltype_cl_coarse2 %in% c('mesenchyme', 'Astrocytes') ~ 'other'`), and they load a Seurat object literally named `asd_chong_annot_wg_ge_v3.1noastro_srt.rds` for the inhibitory (ganglionic eminence) trajectory. Their pseudotime is CellRank `velocity_pseudotime` per lineage, stored in `data/velocity/ge_opc_cellrank_probs.tsv` and `ctx_cellrank_probs.tsv`. Those TSVs are not in the public deposit, so we cannot reproduce their values directly; we use DPT here for methodological consistency with our mouse-SVZ pipeline. Empirical verification: with Astrocytes excluded, DPT produces a clean trajectory (rho = 0.886 against canonical celltype order, full-range pseudotime [0, 1]) on the full 24K-cell lineage; including Astrocytes compresses non-Astrocyte pseudotime to ~0.91 because batch-affected Astrocytes from super-batch 213 dominate the diffusion-graph tip.

The samples are integrated with Harmony (harmonypy, key `sample`) and the DPT graph is built on the corrected embedding (`X_pca_harmony`). Each sample is entirely induced or entirely uninduced (14 and 8 samples), so this integration also aligns induced with uninduced cells. The diagnostic UMAP below uses the uncorrected PCA for comparison. Harmony keyed on the capture super-batch (137xxx, 206xxx, 213xxx) does not resolve the separation of the Astrocytes (KMeans purity 86% before and after correction), which is a further reason to exclude them.

The propagation step is in `08_260502-propagate_pseudotime_to_UTR.ipynb`. The pseudotime is then used by the pseudotime test (`analysis/organoids/pseudotime/02_260503-dirichlet_multinomial_analysis_optimized_2.R`). Astrocytes have no pseudotime after propagation and are dropped by the test.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UMI")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import harmonypy as hm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr
import warnings

warnings.filterwarnings("ignore")

LAYOUT_DPI = 300
FONT_LABEL = 7
FONT_TICK = 6
FONT_LEGEND = 5.5
FONT_LEGEND_TITLE = 6
FONT_ANNOT = 6

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": LAYOUT_DPI,
        "font.size": FONT_LABEL,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": FONT_LABEL,
        "axes.labelsize": FONT_LABEL,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)

sc.settings.verbosity = 1
print("Setup complete.")

In [ ]:
LINEAGE = "Inhibitory neurons"
LINEAGE_TAG = "inhibitory"
ROOT_CELLTYPE = "vRG"  # not "Astrocytes": Li et al. 2023 exclude Astrocytes (see top markdown)
CANONICAL_ORDER = [
    "vRG",
    "ccvRG",
    "INP",
    "LGE_IN",
    "CGE_LGE_IN",
    "CGE_IN",
]  # Astrocytes dropped, as in Li et al. 2023
RANDOM_STATE = 0
N_PCS = 30
N_NEIGHBORS = 30
N_DIFFMAP_COMPS = 15
N_DCS = 10

BASEPATH = f"{DATA_ROOT}/3utr/organoids/knoblich"
COUNTS_PATH = f"{BASEPATH}/cellranger"
COUNTS_FILE = f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized.h5ad"
OUT_LINEAGE_FILE = (
    f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized_"
    f"{LINEAGE_TAG}_dpt.h5ad"
)

FIG_DIR = Path(f"{COUNTS_PATH}/figures/pseudotime_{LINEAGE_TAG}")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Lineage: {LINEAGE}")
print(f"Root celltype: {ROOT_CELLTYPE}")
print(f"Canonical order: {CANONICAL_ORDER}")
print(f"Output checkpoint: {OUT_LINEAGE_FILE}")
print(f"Figures: {FIG_DIR}")

## Load data


In [ ]:
adata = sc.read_h5ad(COUNTS_FILE)
display(adata)

assert "lineage" in adata.obs.columns, "Missing 'lineage' column."
assert "celltype" in adata.obs.columns, "Missing 'celltype' column."
assert adata.obs["lineage"].isna().sum() == 0, "Found NaN in obs.lineage."
assert adata.obs["celltype"].isna().sum() == 0, "Found NaN in obs.celltype."
assert (
    "highly_variable" in adata.var.columns
), "Missing 'highly_variable' flag in var. Re-run HVG selection upstream."
print("Sanity checks passed.")

In [ ]:
print(f"Number of samples: {adata.obs.shape[0]}")

In [ ]:
print("Breakdown of samples:")
print(f"Lineages: {adata.obs['lineage'].value_counts()}")
print(f"Celltypes: {adata.obs['celltype'].value_counts()}")

In [ ]:
print("Breakdown of samples:")
print(f"Induced: {adata.obs['induced'].value_counts()}")
print(f"Batch: {adata.obs['sample'].value_counts()}")

In [ ]:
print(f"Lineages: {adata.obs.groupby(['induced', 'lineage']).size()}")

In [ ]:
adata.obs.groupby(["induced", "lineage", "celltype"]).size().to_frame()

In [ ]:
print(f"{adata.obs.groupby(["induced", "lineage", "sample", "celltype"]).size()}")

## Subset to lineage and HVG-restrict (excluding Astrocytes)

Subset to the inhibitory lineage and drop Astrocytes (mirroring `asd_chong_annot_wg_ge_v3.1noastro_srt.rds` in the published code). `X` is already z-scored from upstream scaling. Re-scaling on a smaller pool would destabilise the variance estimates. PCA is run directly on the existing scaled HVG matrix.


In [ ]:
adata_lin = adata[adata.obs["lineage"] == LINEAGE].copy()
adata_lin = adata_lin[adata_lin.obs["celltype"] != "Astrocytes"].copy()
adata_lin.obs["celltype"] = adata_lin.obs["celltype"].cat.remove_unused_categories()

unexpected = set(adata_lin.obs["celltype"].cat.categories) - set(CANONICAL_ORDER)
assert (
    not unexpected
), f"Lineage subset contains celltypes not in CANONICAL_ORDER: {unexpected}"

adata_lin = adata_lin[:, adata_lin.var["highly_variable"]].copy()

print(f"Lineage subset shape (Astrocytes excluded, HVG-restricted): {adata_lin.shape}")
print(f"Celltype counts:")
print(adata_lin.obs["celltype"].value_counts().reindex(CANONICAL_ORDER).to_string())
print(f"\nInduced state counts:")
print(adata_lin.obs["induced"].value_counts().to_string())

In [ ]:
adata_lin.obs.groupby(["induced", "sample"]).size()

In [ ]:
counts_df = adata_lin.obs.groupby(["sample", "celltype"]).size().to_frame("counts")
has_zeros = (counts_df["counts"] == 0).any()

print(f'Any zeros in column "counts"? {has_zeros}')

## PCA


In [ ]:
sc.pp.pca(adata_lin, n_comps=N_PCS, random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(
    np.arange(1, N_PCS + 1), adata_lin.uns["pca"]["variance_ratio"], "o-", lw=0.8, ms=3
)
ax.set_xlabel("PC")
ax.set_ylabel("Variance ratio")
ax.set_title(f"{LINEAGE}: PCA variance ratio")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_variance_ratio.pdf", bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.pca(
    adata_lin,
    color=["celltype", "sample", "induced"],
    components=["1,2"],
    ncols=3,
    save=False,
    show=True,
)

## Batch effect diagnostic (no Harmony)

Build a temporary k-NN graph + UMAP from the uncorrected PCA. Inspect whether cells of the same celltype and same `induced` state but different `sample` ids are intermixed (good) or form per-sample sub-clusters (bad: would require a different strategy).

This UMAP is a diagnostic only; the DPT graph is computed separately below with `method='gauss'`.


In [ ]:
sc.pp.neighbors(
    adata_lin,
    n_neighbors=N_NEIGHBORS,
    use_rep="X_pca",
    n_pcs=N_PCS,
    random_state=RANDOM_STATE,
)
sc.tl.umap(adata_lin, random_state=RANDOM_STATE)

sc.pl.umap(
    adata_lin,
    color=["celltype", "induced", "sample"],
    ncols=3,
    save=False,
    show=True,
)

In [ ]:
# Per-induced UMAP overlays so within-induced batch heterogeneity is easy to inspect
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, ind_state in zip(axes, [True, False]):
    mask = (adata_lin.obs["induced"].astype(str) == str(ind_state)).values
    if mask.sum() == 0:
        ax.set_title(f"induced={ind_state}: no cells")
        continue
    coords = adata_lin.obsm["X_umap"][mask]
    samples = adata_lin.obs["sample"].values[mask]
    for s in pd.unique(samples):
        m = samples == s
        ax.scatter(
            coords[m, 0], coords[m, 1], s=2, alpha=0.5, label=str(s), rasterized=True
        )
    ax.set_title(
        f"induced={ind_state}: {mask.sum()} cells, {pd.unique(samples).size} samples"
    )
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(fontsize=FONT_LEGEND, markerscale=2, frameon=False, ncol=2, loc="best")
plt.tight_layout()
plt.savefig(FIG_DIR / "diagnostic_umap_per_induced_by_sample.pdf", bbox_inches="tight")
plt.show()

# Harmony batch integration


In [ ]:
# Run Harmony on adata_lin using obs["sample"] as the batch key.
# Harmony adjusts the PCA embedding to remove between-sample batch effects.
# The corrected embedding is stored in obsm["X_pca_harmony"] and used for all
# downstream neighbor graphs (DPT + UMAP).

X_pca = adata_lin.obsm["X_pca"].astype(np.float64)
ho = hm.run_harmony(X_pca, adata_lin.obs, "sample", random_state=RANDOM_STATE)

Z = ho.Z_corr
# harmonypy stores Z_corr as (n_PCs × n_cells); transpose if needed
if Z.shape[0] != adata_lin.n_obs:
    Z = Z.T

adata_lin.obsm["X_pca_harmony"] = Z
print(f"Harmony done. X_pca_harmony shape: {adata_lin.obsm['X_pca_harmony'].shape}")

In [ ]:
# Diagnostic UMAP from the Harmony-corrected embedding.
# Compare with the pre-Harmony UMAP above to judge how much batch mixing improved.
sc.pp.neighbors(
    adata_lin,
    n_neighbors=N_NEIGHBORS,
    use_rep="X_pca_harmony",
    n_pcs=N_PCS,
    random_state=RANDOM_STATE,
)
sc.tl.umap(adata_lin, random_state=RANDOM_STATE)

sc.pl.umap(
    adata_lin,
    color=["celltype", "sample", "induced"],
    ncols=3,
    save=False,
    show=True,
)

## DPT graph

Build a Gaussian-kernel neighbors graph (required by DPT) on the Harmony-corrected PCA (`X_pca_harmony`), then a diffusion map. This overwrites the diagnostic neighbors graph from the previous step.


In [ ]:
sc.pp.neighbors(
    adata_lin,
    n_neighbors=N_NEIGHBORS,
    use_rep="X_pca_harmony",
    n_pcs=N_PCS,
    method="gauss",
    random_state=RANDOM_STATE,
)
sc.tl.diffmap(adata_lin, n_comps=N_DIFFMAP_COMPS)

sc.pl.diffmap(adata_lin, color="celltype", components=["1,2"], save=False, show=True)

## Root cell selection

Two candidate root cells are picked at the extremes of PC1, restricted to `vRG` (ventricular radial glia, the most-undifferentiated celltype after Astrocyte exclusion). Both candidates are visualized on UMAP; pick the one that lies on the periphery furthest from the most-mature celltypes (CGE_IN, CGE_LGE_IN). If neither tip looks correct, switch to PC2 (uncomment the alternative line).


In [ ]:
mask_root = (adata_lin.obs["celltype"] == ROOT_CELLTYPE).values
print(f"Root celltype '{ROOT_CELLTYPE}': {mask_root.sum()} cells")

pc_for_root = 0  # change to 1 to use PC2 if PC1 does not separate the root tip cleanly
pc_vals = adata_lin.obsm["X_pca_harmony"][:, pc_for_root]

iroot_min = int(np.where(mask_root)[0][np.argmin(pc_vals[mask_root])])
iroot_max = int(np.where(mask_root)[0][np.argmax(pc_vals[mask_root])])
print(f"Candidate iroot at PC{pc_for_root + 1} min: {iroot_min}")
print(f"Candidate iroot at PC{pc_for_root + 1} max: {iroot_max}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (label, idx) in zip(axes, [("PC min", iroot_min), ("PC max", iroot_max)]):
    sc.pl.umap(
        adata_lin, color="celltype", ax=ax, show=False, legend_loc="right margin"
    )
    ax.scatter(
        adata_lin.obsm["X_umap"][idx, 0],
        adata_lin.obsm["X_umap"][idx, 1],
        s=80,
        marker="*",
        color="red",
        zorder=10,
        edgecolors="black",
        linewidths=0.8,
    )
    ax.set_title(f"Root candidate: {label} ({ROOT_CELLTYPE} idx {idx})")
plt.tight_layout()
plt.savefig(FIG_DIR / "root_candidates.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (label, idx) in zip(axes, [("PC min", iroot_min), ("PC max", iroot_max)]):
    sc.pl.embedding(
        adata_lin,
        color="celltype",
        basis="X_pca_harmony",
        ax=ax,
        show=False,
        legend_loc="right margin",
    )
    ax.scatter(
        adata_lin.obsm["X_pca_harmony"][idx, 0],
        adata_lin.obsm["X_pca_harmony"][idx, 1],
        s=80,
        marker="*",
        color="red",
        zorder=10,
        edgecolors="black",
        linewidths=0.8,
    )
    ax.set_title(f"Root candidate: {label} ({ROOT_CELLTYPE} idx {idx})")
plt.tight_layout()
plt.savefig(FIG_DIR / "root_candidates.pdf", bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.embedding(
    adata_lin,
    color="celltype",
    basis="X_pca_harmony",
    show=True,
    legend_loc="right margin",
)

sc.pl.embedding(
    adata_lin,
    color="celltype",
    basis="X_pca",
    show=True,
    legend_loc="right margin",
)

In [ ]:
# Pick the root by changing iroot_chosen to either iroot_min or iroot_max after
# inspecting the figure above. Default: PC max (verified by full-data smoke test on the
# astrocyte-excluded inhibitory lineage: PC max of vRG produced a clean trajectory
# rho=0.886 with full-range [0,1] pseudotime; PC min produced rho=-0.886 i.e. backwards).
# Re-run downstream cells if you change it.
iroot_chosen = iroot_max
adata_lin.uns["iroot"] = int(iroot_chosen)
print(f"iroot set to {iroot_chosen}")

## Run DPT


In [ ]:
sc.tl.dpt(adata_lin, n_branchings=0, n_dcs=N_DCS)
print(
    f"DPT done. dpt_pseudotime range: "
    f"[{adata_lin.obs['dpt_pseudotime'].min():.3f}, "
    f"{adata_lin.obs['dpt_pseudotime'].max():.3f}]"
)

## QC plots and rank check


In [ ]:
sc.pl.umap(
    adata_lin,
    color=["dpt_pseudotime", "celltype"],
    color_map="inferno",
    ncols=2,
    save=False,
    show=True,
)
sc.pl.diffmap(
    adata_lin,
    color="dpt_pseudotime",
    components=["1,2"],
    color_map="inferno",
    save=False,
    show=True,
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ordered = adata_lin.obs["celltype"].cat.set_categories(CANONICAL_ORDER, ordered=True)
order_present = [c for c in CANONICAL_ORDER if (ordered == c).any()]
sns.boxplot(
    x=ordered,
    y=adata_lin.obs["dpt_pseudotime"].values,
    order=order_present,
    ax=ax,
    showfliers=False,
)
ax.set_xlabel("Celltype (canonical order)")
ax.set_ylabel("DPT pseudotime")
ax.set_title(f"{LINEAGE}: pseudotime per celltype")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "dpt_per_celltype_box.pdf", bbox_inches="tight")
plt.show()

In [ ]:
ct_means = adata_lin.obs.groupby("celltype", observed=True)["dpt_pseudotime"].median()
ct_means_ordered = ct_means.reindex([c for c in CANONICAL_ORDER if c in ct_means.index])
rho, p = spearmanr(range(len(ct_means_ordered)), ct_means_ordered.values)
print(f"Spearman rho (canonical celltype order vs DPT median) = {rho:.3f}, p = {p:.2e}")
print(f"Per-celltype median pseudotime:")
print(ct_means_ordered.to_string())

assert rho > 0, (
    f"Pseudotime runs backwards (rho={rho}). Re-pick root at the opposite tip "
    f"(swap iroot_chosen to iroot_max or change pc_for_root) and re-run."
)
if rho < 0.7:
    print(
        f"WARNING: rho={rho:.3f} below the 0.7 pass criterion. "
        f"Investigate before saving."
    )
else:
    print(f"PASS: rho={rho:.3f} >= 0.7")

In [ ]:
# Within-induced rank check: pseudotime ordering should hold separately within
# induced=True and induced=False subgroups (otherwise the induction state is
# dominating the trajectory).
for ind_state in adata_lin.obs["induced"].cat.categories:
    sub = adata_lin.obs[adata_lin.obs["induced"] == ind_state]
    if len(sub) == 0:
        continue
    ct_med = sub.groupby("celltype", observed=True)["dpt_pseudotime"].median()
    ordered_med = ct_med.reindex([c for c in CANONICAL_ORDER if c in ct_med.index])
    if len(ordered_med) < 3:
        print(f"induced={ind_state}: too few celltypes ({len(ordered_med)}), skipping")
        continue
    rho_ind, p_ind = spearmanr(range(len(ordered_med)), ordered_med.values)
    print(
        f"induced={ind_state}: n={len(sub):,}, "
        f"celltypes covered={len(ordered_med)}, "
        f"rho={rho_ind:.3f} (p={p_ind:.2e})"
    )

## Save lineage checkpoint

The output file stores the lineage subset with `dpt_pseudotime` in obs (no lineage suffix yet; the suffix is added by the propagation notebook). It also retains the diffusion map (`obsm['X_diffmap']`), the diffusion-graph neighbors (`obsp['connectivities']`/`['distances']`), and the chosen root index in `uns['iroot']`.


In [ ]:
adata_lin.write_h5ad(OUT_LINEAGE_FILE)
print(f"Saved: {OUT_LINEAGE_FILE}")
print(f"Cells: {adata_lin.n_obs:,}, features (HVG): {adata_lin.n_vars:,}")
print(f"obs columns: {list(adata_lin.obs.columns)}")